In [3]:
# 회원가입/동의관리 interface

import sqlite3
import re
import uuid
from datetime import date
from pathlib import Path
 
 
DB_PATH = Path("../schema/파리바게트.db")
 
# 동의항목 순서 고정 
CONSENT_ITEMS = {
    "서비스 이용약관 동의": True,
    "개인정보 수집·이용 동의": True,
    "마케팅 정보 수신 동의": False,
    "제3자 정보 제공 동의": False,
}
CONSENT_ORDER = list(CONSENT_ITEMS.keys())  # 순서고정
 
 
def get_connection():
    if not DB_PATH.exists():
        raise FileNotFoundError(f"DB 파일을 찾을 수 없습니다: {DB_PATH}")
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON;")
    return conn
 
 
def generate_customer_id():
    return "C" + uuid.uuid4().hex[:8].upper()
 
 
def validate_password(pw):
    if len(pw) < 8:
        return False, "8자 이상이어야 합니다."
    if not re.search(r'[a-zA-Z]', pw):
        return False, "영문자를 포함해야 합니다."
    if not re.search(r'[!@#$%^&*(),.?\":{}|<>_\-]', pw):
        return False, "특수문자를 포함해야 합니다."
    return True, ""
 
 
def check_duplicate_id(conn, username):
    row = conn.execute("SELECT 1 FROM 회원 WHERE 아이디=?", (username,)).fetchone()
    return row is not None
 
 
def get_customer_id_by_username(conn, username):
    row = conn.execute("SELECT 고객ID FROM 회원 WHERE 아이디=?", (username,)).fetchone()
    return row["고객ID"] if row else None
 
 
def fetch_consents_ordered(conn, customer_id):
    """동의내역을 CONSENT_ORDER 순서로 반환"""
    rows = conn.execute(
        "SELECT 동의항목, 동의여부, 동의일자, 철회일자 FROM 동의내역 WHERE 고객ID=?",
        (customer_id,)
    ).fetchall()
    row_map = {r["동의항목"]: r for r in rows}
    return [row_map[item] for item in CONSENT_ORDER if item in row_map]
 
 
# 1. 회원가입 
def register_member(conn):
    print("\n── 신규 회원가입 ──")
 
    while True:
        username = input("아이디 (로그인에 사용): ").strip()
        if not username:
            print("아이디를 입력해주세요.")
            continue
        if check_duplicate_id(conn, username):
            print("이미 사용 중인 아이디입니다. 다른 아이디를 입력해주세요.")
            continue
        print("사용 가능한 아이디입니다.")
        break
 
    name = input("이름: ").strip()
    email = input("이메일: ").strip()
 
    while True:
        password = input("비밀번호 (8자 이상, 영문+특수문자 포함): ").strip()
        ok, msg = validate_password(password)
        if not ok:
            print(f"비밀번호 오류: {msg}")
            continue
        break
 
    phone = input("연락처 (예: 010-1234-5678): ").strip()
 
    print("\n주소 입력")
    sido = input("  시/도 (예: 서울시): ").strip()
    sigungu = input("  시/군/구 (예: 마포구): ").strip()
    detail = input("  상세주소 (예: 홍익로 45): ").strip()
    address = f"{sido} {sigungu} {detail}".strip()
 
    if not all([username, name, email, password, phone]):
        print("필수 항목이 누락되었습니다.")
        return
 
    print("\n── 동의항목 선택 ──")
    consents = {}
    for i, (item, required) in enumerate(CONSENT_ITEMS.items(), 1):
        tag = "[필수]" if required else "[선택]"
        answer = input(f"{i}. {tag} {item} (Y/N): ").strip().upper()
        agreed = (answer == "Y")
        if required and not agreed:
            print(f"'{item}'은 필수 동의 항목입니다. 가입을 취소합니다.")
            return
        consents[item] = agreed
 
    customer_id = generate_customer_id()
    today = date.today().isoformat()
 
    try:
        conn.execute("BEGIN")
        conn.execute(
            "INSERT INTO 고객 (고객ID, 이름) VALUES (?, ?)",
            (customer_id, name)
        )
        conn.execute(
            """INSERT INTO 회원 (고객ID, 아이디, 이메일, 주소, 가입일자, 정보제공동의여부, 연락처, 비밀번호)
               VALUES (?, ?, ?, ?, ?, ?, ?, ?)""",
            (customer_id, username, email, address, today,
             1 if consents.get("마케팅 정보 수신 동의") else 0, phone, password)
        )
        for item in CONSENT_ORDER:
            agreed = consents[item]
            conn.execute(
                """INSERT INTO 동의내역 (고객ID, 동의항목, 동의일자, 동의여부, 철회일자)
                   VALUES (?, ?, ?, ?, NULL)""",
                (customer_id, item, today, 1 if agreed else 0)
            )
        conn.commit()
        print(f"\n가입 완료! 아이디: {username} / 고객 ID: {customer_id}")
    except sqlite3.Error as e:
        conn.rollback()
        print(f"가입 실패: {e}")
 
 
# 2. 동의내역 조회
def view_consent(conn):
    print("\n── 동의내역 조회 ──")
    username = input("아이디: ").strip()
 
    customer_id = get_customer_id_by_username(conn, username)
    if not customer_id:
        print("존재하지 않는 아이디입니다.")
        return
 
    rows = fetch_consents_ordered(conn, customer_id)
    if not rows:
        print("동의내역이 없습니다.")
        return
 
    print(f"\n{'번호':<4} {'동의항목':<25} {'동의여부':<8} {'동의일자':<12} {'철회일자'}")
    print("-" * 65)
    for i, r in enumerate(rows, 1):
        status = "동의" if r["동의여부"] else "미동의/철회"
        print(f"{i:<4} {r['동의항목']:<25} {status:<8} {r['동의일자']:<12} {r['철회일자'] or '-'}")
 
 
# 3. 동의 변경/철회
def update_consent(conn):
    print("\n── 동의 변경/철회 ──")
    username = input("아이디: ").strip()
 
    customer_id = get_customer_id_by_username(conn, username)
    if not customer_id:
        print("존재하지 않는 아이디입니다.")
        return
 
    rows = fetch_consents_ordered(conn, customer_id)
    if not rows:
        print("동의내역이 없습니다.")
        return
 
    print("\n현재 동의 상태:")
    for i, r in enumerate(rows, 1):
        status = "동의" if r["동의여부"] else "철회됨"
        print(f"  {i}. {r['동의항목']} - {status}")
 
    try:
        choice = int(input("\n변경할 항목 번호: ").strip()) - 1
        if not (0 <= choice < len(rows)):
            print("잘못된 번호입니다.")
            return
    except ValueError:
        print("숫자를 입력하세요.")
        return
 
    item = rows[choice]["동의항목"]
    action = input(f"'{item}' 을 (1) 동의 / (2) 철회: ").strip()
    today = date.today().isoformat()
 
    if action == "1":
        conn.execute(
            "UPDATE 동의내역 SET 동의여부=1, 동의일자=?, 철회일자=NULL WHERE 고객ID=? AND 동의항목=?",
            (today, customer_id, item)
        )
        conn.commit()
        print("동의 처리 완료.")
    elif action == "2":
        if CONSENT_ITEMS.get(item):
            print("필수 항목은 철회할 수 없습니다.")
            return
        conn.execute(
            "UPDATE 동의내역 SET 동의여부=0, 철회일자=? WHERE 고객ID=? AND 동의항목=?",
            (today, customer_id, item)
        )
        conn.commit()
        print("철회 처리 완료. (이력 보존됨)")
    else:
        print("잘못된 입력입니다.")
 
 
# 4. 회원 탈퇴 
def withdraw_member(conn):
    print("\n── 회원 탈퇴 ──")
    username = input("아이디: ").strip()
 
    row = conn.execute(
        "SELECT 회원.고객ID, 고객.이름 FROM 회원 JOIN 고객 ON 회원.고객ID=고객.고객ID WHERE 아이디=?",
        (username,)
    ).fetchone()
 
    if not row:
        print("존재하지 않는 아이디입니다.")
        return
 
    confirm = input(f"'{row['이름']}' 회원을 탈퇴 처리하시겠습니까? (Y/N): ").strip().upper()
    if confirm != "Y":
        print("취소했습니다.")
        return
 
    try:
        conn.execute("BEGIN")
        conn.execute("DELETE FROM 고객 WHERE 고객ID=?", (row["고객ID"],))
        conn.commit()
        print("탈퇴 처리 완료.")
    except sqlite3.Error as e:
        conn.rollback()
        print(f"탈퇴 실패: {e}")
 
 
# 메인 메뉴 
def main():
    print("회원가입/동의관리 인터페이스 실행")
    try:
        conn = get_connection()
        print(f"DB 접속 완료: {DB_PATH}")
    except FileNotFoundError as e:
        print(e)
        return
 
    while True:
        print("\n" + "=" * 50)
        print("회원가입/동의관리 - 파리바게트 시스템")
        print("=" * 50)
        print("1. 신규 회원가입")
        print("2. 동의내역 조회")
        print("3. 동의 변경/철회")
        print("4. 회원 탈퇴")
        print("0. 종료")
        print("=" * 50)
 
        choice = input("메뉴 선택 > ").strip()
 
        try:
            if choice == "1":
                register_member(conn)
            elif choice == "2":
                view_consent(conn)
            elif choice == "3":
                update_consent(conn)
            elif choice == "4":
                withdraw_member(conn)
            elif choice == "0":
                break
            else:
                print("잘못된 메뉴 번호입니다.")
        except sqlite3.Error as e:
            print(f"SQLite 오류: {e}")
        except Exception as e:
            print(f"오류: {e}")
 
    conn.close()
    print("종료합니다.")
 
 
if __name__ == "__main__":
    main()

회원가입/동의관리 인터페이스 실행
DB 접속 완료: ../schema/파리바게트.db

회원가입/동의관리 - 파리바게트 시스템
1. 신규 회원가입
2. 동의내역 조회
3. 동의 변경/철회
4. 회원 탈퇴
0. 종료


메뉴 선택 >  0


종료합니다.
